## 21.0 Setup + settings

In [79]:
import re, json
from collections import Counter
import pandas as pd
from IPython.display import display
from src.paths import SECTIONS, LETTERS_JSONL, PAGES_FINAL, CHUNKS

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f]

SECS    = read_jsonl(SECTIONS)
LETTERS = read_jsonl(LETTERS_JSONL)
PAGES   = read_jsonl(PAGES_FINAL)
PRINTED = {p["pdf_page"]: p["printed_page"] for p in PAGES}
METHOD  = {p["pdf_page"]: p["method"] for p in PAGES}
HEADING = {"L0", "L1", "L2", "L3"}

# ---- size settings (characters; ~4-5 chars per token) ----
TARGET     = 1000     # a split section aims for chunks about this size (~220 tokens)
KEEP_WHOLE = 1400     # a section up to this size stays ONE chunk, even above TARGET
HARD_MAX   = 1600     # no chunk may be bigger (a 512-token embedding model fits this)
MIN_CHUNK  = 300      # a sub-heading starts a new chunk only if the current one has at least this much
OVERLAP    = 1        # sentences repeated at the start of the next chunk of the same section

# ---- paragraph gap: learned from the data, not typed ----
_gaps = Counter(round(b["top"] - a["top"]) for s in SECS for a, b in zip(s["lines"], s["lines"][1:])
                if a["level"] == b["level"] == "body" and a["pdf_page"] == b["pdf_page"]
                and a.get("top") is not None and b.get("top") is not None and b["top"] > a["top"])
if not _gaps:
    raise ValueError("sections.jsonl has no 'top' values → re-run 05_sections: 20.3 (with \"top\") → 20.T → 20.H")
LINE_STEP = _gaps.most_common(1)[0][0]           # normal distance between two lines of one paragraph
PARA_GAP  = LINE_STEP * 1.3                      # clearly bigger than that = new paragraph
print(f"{len(SECS)} sections · {len(LETTERS)} letters · line step {LINE_STEP} → PARA_GAP {PARA_GAP:.1f}")
print("most common gaps:", _gaps.most_common(6))

106 sections · 21 letters · line step 14 → PARA_GAP 18.2
most common gaps: [(14, 1665), (19, 632), (23, 261), (13, 33), (46, 25), (20, 13)]


## 21.1 Text helpers


In [80]:
def join_lines(texts):
    """Visual lines → running text. 'knowledge-' + 'based' → 'knowledge-based'; otherwise a space."""
    out = ""
    for t in texts:
        t = t.strip()
        if not t:
            continue
        out = out + t if re.search(r"[A-Za-z]-$", out) else (out + " " + t).strip()
    return out

# dots that do NOT end a sentence: abbreviations and single initials (C.K., D.O.)
NO_END = re.compile(r"(\b(?:No|Nos|e\.g|i\.e|Rs|Dr|Mr|Ms|Mrs|Smt|Shri|Sh|etc|viz|vs|Govt|Dept|Fig|approx|Sl|para)\.|\b[A-Za-z]\.)$",
                    re.I)

def split_sentences(text):
    """Split at '. ' / '? ' / '! ' before a capital or digit, then glue back pieces cut after an abbreviation."""
    parts = re.split(r"(?<=[.!?])\s+(?=[\"'(\[]?[A-Z0-9])", text.strip())
    out = []
    for p in parts:
        if out and NO_END.search(out[-1]):
            out[-1] += " " + p
        else:
            out.append(p)
    return [s for s in out if s]

def hard_split(text, size):
    """Last resort for one sentence longer than HARD_MAX: cut between words."""
    words, out, cur = text.split(), [], ""
    for w in words:
        if cur and len(cur) + 1 + len(w) > size:
            out.append(cur); cur = w
        else:
            cur = (cur + " " + w).strip()
    return out + ([cur] if cur else [])

LIST_START = re.compile(r"^(-\s|\(?[ivx]{1,4}\)\s|\(?[a-h]\)\s|[a-h]\.\s|\d{1,2}[.)]\s)")
LIST_MARK  = re.compile(r"^(-\s|\(?[a-z]{1,4}[.)]\s|e\.g\.,?\s)", re.I)              # a) · c. · iii. · e.g. · "- "
LABEL_BODY = re.compile(r"^((chapter|annexure)(\s*[-:]?\s*[0-9ivx]+)?|[ivx]+)\s*$", re.I)   # "Annexure" · "I" · "III"

def glue_continuations(blocks):
    """A paragraph that starts lower-case (not a list item) right after a paragraph that does not end a
    sentence is the SAME sentence, broken by a page break or a big line gap → join them back."""
    out = []
    for b in blocks:
        prev = out[-1] if out else None
        if (prev and b["kind"] == "para" and prev["kind"] == "para"
                and re.match(r"[a-z]", b["text"]) and not LIST_MARK.match(b["text"])
                and not re.search(r"[.?!]$", prev["text"])):
            out[-1] = {**prev, "text": join_lines([prev["text"], b["text"]]), "pages": prev["pages"] | b["pages"]}
        else:
            out.append(b)
    return out

## 21.1 Text helpers

In [81]:
def join_lines(texts):
    """Visual lines → running text. 'knowledge-' + 'based' → 'knowledge-based'; otherwise a space."""
    out = ""
    for t in texts:
        t = t.strip()
        if not t:
            continue
        out = out + t if re.search(r"[A-Za-z]-$", out) else (out + " " + t).strip()
    return out

# dots that do NOT end a sentence: abbreviations and single initials (C.K., D.O.)
NO_END = re.compile(r"(\b(?:No|Nos|e\.g|i\.e|Rs|Dr|Mr|Ms|Mrs|Smt|Shri|Sh|etc|viz|vs|Govt|Dept|Fig|approx|Sl|para)\.|\b[A-Za-z]\.)$",
                    re.I)

def split_sentences(text):
    """Split at '. ' / '? ' / '! ' before a capital or digit, then glue back pieces cut after an abbreviation."""
    parts = re.split(r"(?<=[.!?])\s+(?=[\"'(\[]?[A-Z0-9])", text.strip())
    out = []
    for p in parts:
        if out and NO_END.search(out[-1]):
            out[-1] += " " + p
        else:
            out.append(p)
    return [s for s in out if s]

def hard_split(text, size):
    """Last resort for one sentence longer than HARD_MAX: cut between words."""
    words, out, cur = text.split(), [], ""
    for w in words:
        if cur and len(cur) + 1 + len(w) > size:
            out.append(cur); cur = w
        else:
            cur = (cur + " " + w).strip()
    return out + ([cur] if cur else [])

LIST_START = re.compile(r"^(-\s|\(?[ivx]{1,4}\)\s|\(?[a-h]\)\s|[a-h]\.\s|\d{1,2}[.)]\s)")

## 21.2 Section lines → blocks

In [82]:
def section_blocks(sec):
    """Lines → blocks in order: heading / para / list / table.
    A caption is glued to the block after it. Diagram lines (PDF 32) become one list block."""
    blocks, buf, buf_pages, prev = [], [], set(), None

    def flush():
        nonlocal buf, buf_pages
        if buf:
            blocks.append({"kind": "para", "text": join_lines(buf), "pages": set(buf_pages)})
        buf, buf_pages = [], set()

    for l in sec["lines"]:
        lv, t, n = l["level"], l["text"], l["pdf_page"]
        if lv in HEADING or LABEL_BODY.match(t.strip()):               # bare "Annexure" / "I" = part of a title
            flush(); blocks.append({"kind": "heading", "text": t.strip(), "pages": {n}})
        elif lv == "table":
            flush(); blocks.append({"kind": "table", "text": t.strip(), "pages": {n}})
        elif lv == "caption":
            flush(); blocks.append({"kind": "caption", "text": t.strip(), "pages": {n}})
        elif lv == "diagram":
            flush()
            if blocks and blocks[-1]["kind"] == "list" and blocks[-1].get("diagram"):
                blocks[-1]["text"] += "\n" + t.strip(); blocks[-1]["pages"].add(n)
            else:
                blocks.append({"kind": "list", "text": t.strip(), "pages": {n}, "diagram": True})
        else:                                                    # body (and anything else)
            new_para = (not buf or prev is None
                        or LIST_START.match(t.strip())
                        or (n != prev["pdf_page"] and re.search(r"[.:;?!]$", buf[-1]))
                        or (n == prev["pdf_page"] and l.get("top") is not None and prev.get("top") is not None
                            and not (0 < l["top"] - prev["top"] <= PARA_GAP)))
            if new_para:
                flush()
            buf.append(t); buf_pages.add(n)
        prev = l
    flush()

    # glue each caption onto the block that follows it ("Exhibit 1: …" + the 8 steps)
    out = []
    for b in blocks:
        if out and out[-1]["kind"] == "caption" and b["kind"] in ("para", "list"):
            cap = out.pop()
            b = {**b, "text": cap["text"] + "\n" + b["text"], "pages": cap["pages"] | b["pages"]}
        out.append(b)
    for b in out:                                                 # a caption with nothing after it → plain para
        if b["kind"] == "caption":
            b["kind"] = "para"
    return glue_continuations(out)

## 21.3  Blocks → chunks (pack)

In [83]:
SMALL = 200           # a chunk under this is merged into its neighbour (same section/letter) when it fits

def _size(parts):
    return sum(len(p) + 2 for p in parts) - 2 if parts else 0

def merge_small(chunks):
    """Tiny chunks (a leftover tail, 'Checked by / Verified by') join the chunk before them;
    a tiny FIRST chunk joins the one after it. Only when the result still fits HARD_MAX."""
    body = lambda c: c["parts"][1:] if c.get("overlap") else c["parts"]   # drop the repeated overlap sentence
    out = []
    for c in chunks:
        if out and _size(body(c)) < SMALL and _size(out[-1]["parts"]) + 2 + _size(body(c)) <= HARD_MAX:
            out[-1]["parts"] += body(c); out[-1]["pages"] |= c["pages"]
        else:
            out.append(c)
    if len(out) > 1 and _size(out[0]["parts"]) < SMALL and _size(out[0]["parts"]) + 2 + _size(body(out[1])) <= HARD_MAX:
        first, nxt = out.pop(0), out[0]
        out[0] = {**nxt, "parts": first["parts"] + body(nxt), "pages": first["pages"] | nxt["pages"],
                  "sub_heading": first["sub_heading"], "overlap": ""}
    return out

def pieces_of(block):
    """A block that fits TARGET stays whole; a longer one is cut into sentences (never inside one)."""
    if len(block["text"]) <= TARGET or block["kind"] == "list":
        return [block["text"]]
    out = []
    for s in split_sentences(block["text"]):
        out += hard_split(s, TARGET) if len(s) > HARD_MAX else [s]
    return out

def pack(blocks, keep_whole=True):
    """Blocks of ONE section/letter → list of {"parts": [...], "pages": set, "sub_heading": str|None}."""
    body = [b for b in blocks if b["kind"] in ("para", "list")]
    if not body:
        return []                                                 # heading-only section → no chunk
    if keep_whole and sum(len(b["text"]) for b in blocks if b["kind"] != "table") <= KEEP_WHOLE:
        keep = [b for b in blocks if b["kind"] != "table"]
        return [{"parts": [b["text"] for b in keep], "pages": set().union(*(b["pages"] for b in keep)),
                 "sub_heading": None, "overlap": ""}]

    chunks, heading, title_seen = [], None, False

    def start(overlap=""):
        return {"parts": [overlap] if overlap else [], "pages": set(), "sub_heading": heading,
                "overlap": overlap, "has_body": False}

    def size(c):
        return sum(len(p) + 2 for p in c["parts"])

    cur = start()
    for b in blocks:
        if b["kind"] == "table":
            continue                                               # tables get their own chunks (21.4)
        if b["kind"] == "heading":
            is_title, title_seen = not title_seen, True            # first heading = the section's own title
            if not is_title:
                heading = b["text"]
                if cur["has_body"] and size(cur) >= MIN_CHUNK:     # sub-heading → new chunk, no overlap
                    chunks.append(cur)
                    cur = start()
                elif not cur["has_body"]:
                    cur["sub_heading"] = heading
            cur["parts"].append(b["text"]); cur["pages"] |= b["pages"]
            continue
        title_seen = True
        for piece in pieces_of(b):
            if cur["has_body"] and size(cur) + len(piece) > TARGET:
                last = split_sentences(cur["parts"][-1])[-OVERLAP:] if OVERLAP else []
                ov = " ".join(last) if last and len(" ".join(last)) <= TARGET // 3 else ""
                chunks.append(cur)
                cur = start(ov if len(ov) + 2 + len(piece) <= HARD_MAX else "")   # no room → skip the overlap
            cur["parts"].append(piece); cur["pages"] |= b["pages"]; cur["has_body"] = True
    if cur["has_body"]:
        chunks.append(cur)
    elif chunks and cur["parts"]:                                  # only a trailing heading left → previous chunk
        chunks[-1]["parts"] += cur["parts"]; chunks[-1]["pages"] |= cur["pages"]
    return merge_small(chunks)

## 21.4 Tables

In [84]:
def table_rows(t):
    rows = t.get("texts") or []
    return [r.strip() for r in rows if str(r).strip()]

def table_chunks(t):
    """Title + header + rows; a big table → groups of rows, title + header repeated in every group.
    A single row too long for one chunk is cut between words (never truncated)."""
    head = t.get("title") or f"Table {t['id']}"
    cells = [str(h) for h in (t.get("header") or []) if h and not re.fullmatch(r"col\d+", str(h))]
    hdr = " | ".join(cells)
    top = head + (f"\n{hdr}" if hdr and t.get("kind") != "2_col_list" else "")
    room = max(TARGET - len(top) - 1, 200)                 # space left for rows under the repeated title/header
    rows = []
    for r in table_rows(t):
        rows += [r] if len(r) <= room else hard_split(r, room)
    groups, cur = [], []
    for r in rows:
        if cur and len(top) + sum(len(x) + 1 for x in cur) + len(r) > TARGET:
            groups.append(cur); cur = []
        cur.append(r)
    if cur or not groups:
        groups.append(cur)
    return [top + ("\n" + "\n".join(g) if g else "") for g in groups]

## 21.5 Build every chunk

In [85]:
LETTER_PARENT = "Annexure II: Compendium of Letters"

def record(chunk_id, doc_type, path, sub_heading, pages, text, source, **extra):
    pages = sorted(pages)
    prefix = " > ".join(path) + (f" > {sub_heading}" if sub_heading else "")
    return {"chunk_id": chunk_id, "doc_type": doc_type, "section_id": extra.get("section_id"),
            "letter_no": extra.get("letter_no"), "path": path, "sub_heading": sub_heading,
            "pdf_pages": pages, "printed_pages": [PRINTED.get(n) for n in pages], "source": source,
            "text": text, "embed_text": f"{prefix}\n\n{text}", "n_chars": len(text),
            "footnotes": extra.get("footnotes", [])}

def section_chunks(sec):
    out, src = [], "ocr" if any(METHOD.get(n) == "ocr" for n in sec["pdf_pages"]) else "text"
    for k, c in enumerate(pack(section_blocks(sec)), 1):
        notes = [f for f in sec["footnotes"] if f["pdf_page"] in c["pages"]]
        out.append(record(f"{sec['section_id']}-{k:02d}", "glossary" if sec["kind"] == "glossary" else "section",
                          sec["path"], c["sub_heading"], c["pages"], "\n\n".join(c["parts"]), src,
                          section_id=sec["section_id"], footnotes=notes))
    for t in sec["tables"]:
        for k, text in enumerate(table_chunks(t), 1):
            out.append(record(f"T{t['id']}-{k:02d}", "table", sec["path"], t.get("title"), {t["pdf_page"]},
                              text, "text", section_id=sec["section_id"]))
    return out

def letter_chunks(L):
    no, path = L["letter_no"], [LETTER_PARENT, f"Letter {L['letter_no']} ({L['date_text']}): {L['subject']}"]
    if L["status"] != "ok":
        text = f"Letter {no} dated {L['date_text']}: {L['subject']}. {L['notice']}"
        return [record(f"L{no:02d}-01", "letter_stub", path, None, L["pdf_pages"], text, "ocr", letter_no=no)]
    blocks = glue_continuations([{"kind": "para", "text": p.strip(), "pages": set(L["pdf_pages"])}
                                 for p in L["body"].split("\n\n") if p.strip()])
    return [record(f"L{no:02d}-{k:02d}", "letter", path, None, c["pages"], "\n\n".join(c["parts"]), "ocr",
                   letter_no=no)
            for k, c in enumerate(pack(blocks), 1)]

CHUNKS_OUT = [c for s in SECS for c in section_chunks(s)] + [c for L in LETTERS for c in letter_chunks(L)]
print(f"{len(CHUNKS_OUT)} chunks:", dict(Counter(c["doc_type"] for c in CHUNKS_OUT)))

466 chunks: {'table': 82, 'section': 314, 'letter': 68, 'letter_stub': 2}


## 21.6 Look (changes nothing)

In [86]:
sizes = pd.Series([c["n_chars"] for c in CHUNKS_OUT])
print("chars per chunk:", sizes.describe().round(0).to_dict())
print("by size:", pd.cut(sizes, [0, 300, 600, 1000, 1400, 1600, 10**6]).value_counts(sort=False).to_dict())

def show_chunks(section_number=None, letter_no=None, k=3):
    sid = next((s["section_id"] for s in SECS if s["number"] == section_number), None)
    for c in CHUNKS_OUT:
        if (sid and c["section_id"] == sid) or (letter_no and c["letter_no"] == letter_no):
            print(f"\n── {c['chunk_id']} · {c['doc_type']} · printed {c['printed_pages']} · {c['n_chars']} chars")
            print(c["embed_text"][:600])

show_chunks("5.3")

chars per chunk: {'count': 466.0, 'mean': 762.0, 'std': 263.0, 'min': 96.0, '25%': 574.0, '50%': 817.0, '75%': 948.0, 'max': 1592.0}
by size: {Interval(0, 300, closed='right'): 25, Interval(300, 600, closed='right'): 98, Interval(600, 1000, closed='right'): 285, Interval(1000, 1400, closed='right'): 56, Interval(1400, 1600, closed='right'): 2, Interval(1600, 1000000, closed='right'): 0}

── S023-01 · section · printed ['12'] · 676 chars
Chapter 5: Recruitment > 5.3 Process of Recruitment

5.3. Process of Recruitment

The recruitment process should be relevant, aptitude/ skill-based, transparentand equitable. It should be applied consistently to all candidates.

The key steps in the process of recruitment are shown in Exhibit 1 and explained below:

Exhibit 1: Key steps in the Process of Recruitment
1. Identify and validate vacancies.
2. Review Job Description.
3. Design and disseminate advertisement for vacancy.
4. Decide on stages of selection and special weightage criteria, if any.
5

## 21.H TEST

In [87]:
n_pass, report = -1, []                 # a crash below can never leave an old "all passed" behind
TESTS = []
def check(group, name, get_actual, expected, desc=None):
    try:
        actual = get_actual()
        ok = expected(actual) if callable(expected) else actual == expected
    except Exception as e:
        actual, ok = f"ERROR {type(e).__name__}: {e}", False
    TESTS.append({"group": group, "test": name, "result": "PASS" if ok else "FAIL",
                  "expected": desc if callable(expected) else expected, "actual": actual})

flat = lambda s: re.sub(r"\s+", " ", s).strip()          # compare text ignoring line breaks
BY_SEC = {}
for c in CHUNKS_OUT:
    if c["doc_type"] in ("section", "glossary"):
        BY_SEC.setdefault(c["section_id"], []).append(c)
ANY_CHUNK = {c["section_id"] for c in CHUNKS_OUT if c["section_id"]}
PARENTS   = {s["parent_id"] for s in SECS if s["parent_id"]}
SEC_PAGES = {s["section_id"]: set(s["pdf_pages"]) for s in SECS}

def lost_section_text():
    lost = []
    for s in SECS:
        blocks = section_blocks(s)
        if not any(b["kind"] in ("para", "list") for b in blocks):
            continue                                              # heading-only / table-only: no section chunk by design
        joined = flat("\n".join(c["text"] for c in BY_SEC.get(s["section_id"], [])))
        for b in blocks:
            if b["kind"] == "table":
                continue
            for sent in (split_sentences(b["text"]) if b["kind"] == "para" else [b["text"]]):
                if flat(sent) not in joined:
                    lost.append((s["section_id"], sent[:50]))
    return lost[:10]

def lost_letter_text():
    lost = []
    for L in LETTERS:
        if L["status"] != "ok":
            continue
        joined = flat("\n".join(c["text"] for c in CHUNKS_OUT if c["letter_no"] == L["letter_no"]))
        lost += [(L["letter_no"], s[:50]) for p in L["body"].split("\n\n") for s in split_sentences(p)
                 if flat(s) not in joined]
    return lost[:10]

def overlap_misses():
    miss = []
    for chunks in BY_SEC.values():
        for a, b in zip(chunks, chunks[1:]):
            if a["sub_heading"] == b["sub_heading"]:              # same topic, cut by size → overlap expected
                last = split_sentences(a["text"].split("\n\n")[-1])[-1]
                if len(last) <= TARGET // 3 and not b["text"].startswith(last):
                    miss.append(b["chunk_id"])
    return miss

G = "21 coverage"
check(G, "chunk ids unique", lambda: [k for k, v in Counter(c["chunk_id"] for c in CHUNKS_OUT).items() if v > 1], [])
check(G, "sections without any chunk are heading-only parents",
      lambda: [s["section_id"] for s in SECS if s["section_id"] not in ANY_CHUNK and s["section_id"] not in PARENTS], [])
check(G, "nothing lost: every sentence of every section is in its chunks", lost_section_text, [])
check(G, "every table in a table chunk",
      lambda: sorted({c["chunk_id"][1:].rsplit("-", 1)[0] for c in CHUNKS_OUT if c["doc_type"] == "table"}),
      sorted(t["id"] for s in SECS for t in s["tables"]))
check(G, "nothing lost: every table row starts somewhere in its table's chunks",
      lambda: [(t["id"], r[:40]) for s in SECS for t in s["tables"] for r in table_rows(t)
               if flat(r[:60]) not in flat("\n".join(table_chunks(t)))], [])
check(G, "19 readable letters chunked + 2 stubs (11, 14)",
      lambda: (len({c["letter_no"] for c in CHUNKS_OUT if c["doc_type"] == "letter"}),
               sorted(c["letter_no"] for c in CHUNKS_OUT if c["doc_type"] == "letter_stub")),
      (19, [11, 14]))
check(G, "nothing lost: every sentence of every readable letter", lost_letter_text, [])

G = "21 sizes"
check(G, "no empty chunk", lambda: [c["chunk_id"] for c in CHUNKS_OUT if not c["text"].strip()], [])
check(G, "no chunk that is only a heading",
      lambda: [c["chunk_id"] for c in CHUNKS_OUT if c["doc_type"] == "section" and len(c["text"].split("\n\n")) == 1
               and c["text"].strip() in {b["text"] for s in SECS if s["section_id"] == c["section_id"]
                                          for b in section_blocks(s) if b["kind"] == "heading"}], [])
check(G, f"no small chunk (< {SMALL}) next to a neighbour it could join",
      lambda: [c["chunk_id"] for c in CHUNKS_OUT if c["doc_type"] in ("section", "letter") and c["n_chars"] < SMALL
               and sum(1 for d in CHUNKS_OUT if d["doc_type"] == c["doc_type"]
                       and (d["section_id"], d["letter_no"]) == (c["section_id"], c["letter_no"])) > 1], [])
check(G, f"no chunk over HARD_MAX ({HARD_MAX})",
      lambda: [(c["chunk_id"], c["n_chars"]) for c in CHUNKS_OUT if c["n_chars"] > HARD_MAX], [])
check(G, "next chunk of the same topic starts with the overlap sentence", overlap_misses, [])

G = "21 citations"
check(G, "chunk pages inside its section's pages",
      lambda: [c["chunk_id"] for c in CHUNKS_OUT if c["section_id"] and not set(c["pdf_pages"]) <= SEC_PAGES[c["section_id"]]], [])
check(G, "every chunk has printed pages",
      lambda: [c["chunk_id"] for c in CHUNKS_OUT if not c["printed_pages"] or None in c["printed_pages"]], [])
check(G, "embed_text = path first, then text",
      lambda: [c["chunk_id"] for c in CHUNKS_OUT
               if not (c["embed_text"].startswith(c["path"][0]) and c["embed_text"].endswith(c["text"]))], [])

G = "21 real cases"
check(G, "PDF 32: caption + all 8 steps in one chunk",
      lambda: len([c for c in CHUNKS_OUT if "Exhibit 1" in c["text"] and "1. Identify and validate vacancies." in c["text"]
                   and "8. Disseminate result and issue offer letter." in c["text"]]), 1)
check(G, "no section chunk that is only a title ('Annexure / I / …')",
      lambda: [c["chunk_id"] for c in CHUNKS_OUT if c["doc_type"] == "section"
               and all(LABEL_BODY.match(p.strip()) or len(p) < 60 and not re.search(r"[.:;?!]", p)
                       for p in c["text"].split("\n\n"))
               and c["n_chars"] < 100], [])
check(G, "no chunk ends with a stray 'Annexure' / 'III' line",
      lambda: [c["chunk_id"] for c in CHUNKS_OUT if c["doc_type"] == "section"
               and re.search(r"\n\n(Annexure|Chapter|[IVX]+)\s*$", c["text"])], [])
check(G, "compound kept: 'knowledge-based'",
      lambda: (any("knowledge-based" in c["text"] for c in CHUNKS_OUT), any("knowledge- based" in c["text"] for c in CHUNKS_OUT)),
      (True, False))

G = "21 unit"
check(G, "join: compound keeps hyphen", lambda: join_lines(["a simpler knowledge-", "based training"]),
      "a simpler knowledge-based training")
check(G, "join: in- + service", lambda: join_lines(["candidates of the in-", "service cadre"]),
      "candidates of the in-service cadre")
check(G, "join: normal lines get a space", lambda: join_lines(["Presence of a", "medical college"]),
      "Presence of a medical college")
check(G, "split: 'D.O. No. 7(45)' is not a sentence end",
      lambda: split_sentences("Refer D.O. No. 7(45)/2014-NRHM dated 19th January. The Ministry agreed."),
      ["Refer D.O. No. 7(45)/2014-NRHM dated 19th January.", "The Ministry agreed."])
check(G, "split: 'e.g.' and 'Rs.' are not sentence ends",
      lambda: split_sentences("Use tests e.g. the written test. Pay Rs. 500 to each. Then stop."),
      ["Use tests e.g. the written test.", "Pay Rs. 500 to each.", "Then stop."])
check(G, "split: initials 'C.K.' are not a sentence end",
      lambda: split_sentences("A letter from C.K. Mishra was sent. Then more."),
      ["A letter from C.K. Mishra was sent.", "Then more."])
check(G, "glue: a lower-case paragraph continues the unfinished one before it",
      lambda: [b["text"] for b in glue_continuations([
          {"kind": "para", "text": "the State should ensure that", "pages": {1}},
          {"kind": "para", "text": "all vacancies are filled.", "pages": {2}},
          {"kind": "para", "text": "a) Background information.", "pages": {2}}])],
      ["the State should ensure that all vacancies are filled.", "a) Background information."])
check(G, "bare 'Annexure' / 'III' body lines count as title, not text",
      lambda: [bool(LABEL_BODY.match(x)) for x in ["Annexure", "I", "III", "In the State"]], [True, True, True, False])
check(G, "hard_split: no piece over the limit", lambda: max(len(p) for p in hard_split("word " * 400, 100)) <= 100, True)
_paras = [{"kind": "para", "text": " ".join(f"Sentence {i}{k} is about recruitment rules." for k in range(15)), "pages": {40 + i}}
          for i in range(3)]
check(G, "pack: 3 long paragraphs → 3 chunks, 2nd starts with the 1st's last sentence",
      lambda: (lambda ch: (len(ch), ch[1]["parts"][0] == split_sentences(ch[0]["parts"][-1])[-1]))(pack(_paras)),
      (3, True))
_long = [{"kind": "para", "text": " ".join(f"Short sentence {k} here." for k in range(40))
                                  + " The last sentence is long " + "and keeps going " * 16 + "to the end.", "pages": {1}},
         {"kind": "para", "text": "Phase-1 " + "designate nodal officers and estimate HR " * 36 + "end.", "pages": {2}}]
check(G, "pack: overlap is dropped when it would push a chunk over HARD_MAX",
      lambda: max(sum(len(p) + 2 for p in c["parts"]) - 2 for c in pack(_long)) <= HARD_MAX, True)
_para = lambda p: {"kind": "para", "text": " ".join(f"Sentence {k} about appraisal rules here." for k in range(25)), "pages": {p}}
_tail = [_para(60), _para(61), {"kind": "para", "text": "Checked by[__]", "pages": {61}},
         {"kind": "para", "text": "Verified by[__]", "pages": {61}}]
check(G, "pack: a tiny tail ('Checked by / Verified by') joins the chunk before it",
      lambda: (lambda ch: (len(ch), ch[-1]["parts"][-1]))(pack(_tail)), (2, "Verified by[__]"))
check(G, "table: 'col1 | col2' header cells are dropped",
      lambda: table_chunks({"id": "y-1", "title": "Form", "header": ["col1", "col2", "Signature"], "kind": "form_grid",
                            "texts": ["Signature of Reporting Officer"], "pdf_page": 1})[0],
      "Form\nSignature\nSignature of Reporting Officer")
_wide = {"id": "z-1", "title": "Salary table", "header": ["Years", "Salary"], "kind": "data_table", "pdf_page": 1,
         "texts": ["- 0-2 years: " + " ".join(f"band{k} Rs. {k}000" for k in range(260))]}
check(G, "table: a very long row is split, not cut off",
      lambda: (lambda g: (len(g) > 1, max(map(len, g)) <= HARD_MAX, "…" in "".join(g),
                          flat(" ".join(x.split("\n", 2)[-1] for x in g)) == flat(_wide["texts"][0])))(table_chunks(_wide)),
      (True, True, False, True))
check(G, "table: big table → groups, title repeated, none over HARD_MAX",
      lambda: (lambda g: (len(g) > 1, all(x.startswith("Big table") for x in g), max(map(len, g)) <= HARD_MAX))(
          table_chunks({"id": "x-1", "title": "Big table", "header": ["A", "B"], "kind": "data_table",
                        "texts": [f"- row {i}: " + "value " * 8 for i in range(60)], "pdf_page": 1})),
      (True, True, True))

report = pd.DataFrame(TESTS)
n_pass = int((report["result"] == "PASS").sum())
print(f"{'✅' if n_pass == len(report) else '❌'}  {n_pass}/{len(report)} tests passed")
failed = report[report["result"] == "FAIL"]
failed if len(failed) else "All tests passed — nothing to show."

✅  34/34 tests passed


'All tests passed — nothing to show.'

## 21.H Save, only when all tests pass

In [88]:
if n_pass == len(report):
    with CHUNKS.open("w", encoding="utf-8") as f:
        for c in CHUNKS_OUT:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")
    assert read_jsonl(CHUNKS) == CHUNKS_OUT, "reload differs from what was saved"
    print(f"saved {len(CHUNKS_OUT)} chunks → {CHUNKS.name} (reload identical)")
else:
    print("NOT saved — fix the failing tests first")

saved 466 chunks → chunks.jsonl (reload identical)



## 21.8 Report

In [89]:
import random
chunks = read_jsonl(CHUNKS)
prose  = [c for c in chunks if c["doc_type"] in ("section", "letter")]
LIST_MARK = re.compile(r"^(\(?[a-z]{1,4}[.)]|e\.g\.,?)\s", re.I)          # a) · c. · iii. · e.g.

checks = {
    "starts lower-case (cut mid-sentence?)": [c for c in prose if re.match(r"[a-z]", c["text"].lstrip("-• "))
                                              and not LIST_MARK.match(c["text"].lstrip("-• "))],
    "ends without . : ; ? ! )":              [c for c in prose if not re.search(r"[.:;?!)\]\"'’”]$", c["text"].rstrip())],
    "same text twice":                       [c for c, n in Counter(c["text"] for c in chunks).items() if n > 1],
    "has a lone heading fragment":           [c for c in prose if any(re.fullmatch(r"(Annexure|Chapter|[IVX]+|\d+)", p.strip())
                                                                    for p in c["text"].split("\n\n"))],
    "mostly non-letters (OCR junk?)":        [c for c in chunks if sum(ch.isalpha() for ch in c["text"]) / max(len(c["text"]), 1) < 0.6],
}
for name, hits in checks.items():
    ids = [h["chunk_id"] if isinstance(h, dict) else h[:40] for h in hits]
    print(f"{len(hits):>4}  {name:<40} {ids[:6]}")

random.seed(11)
sample = [c for t in ("section", "letter", "table") for c in random.sample([x for x in chunks if x["doc_type"] == t], 2)]
sample += [min(prose, key=lambda c: c["n_chars"]), max(prose, key=lambda c: c["n_chars"])]
for c in sample:
    print(f"\n── {c['chunk_id']} · {c['doc_type']} · printed {c['printed_pages']} · {c['n_chars']} chars")
    print(c["embed_text"][:400], "…" if len(c["embed_text"]) > 400 else "")

   3  starts lower-case (cut mid-sentence?)    ['S099-04', 'S099-05', 'L10-09']
  55  ends without . : ; ? ! )                 ['S027-01', 'S072-02', 'S073-27', 'S073-28', 'S073-29', 'S073-30']
   0  same text twice                          []
   1  has a lone heading fragment              ['S090-01']
   0  mostly non-letters (OCR junk?)           []

── S088-05 · section · printed ['83'] · 866 chars
Annexure I: Suggestive Procedural Formats > Redressal Mechanism

The same process should be adopted for the matters that are registered at State Grievance Redressal Committee.

- In case, the District Grievance Redressal Committee fails to resolve the issue, or the grievance is beyond its remit, this should be referred by the nodal person to the State Grievance Redressal Committee within the presc …

── S099-08 · section · printed ['139', '140'] · 492 chars
Annexure III: Learnings from Other States > Person-Centred Care through Shared leadership- Banka Model of Bihar > Programme Challenges

In [90]:
POS = {c["chunk_id"]: k for k, c in enumerate(chunks)}

def boundary(cid, n=160):
    k = POS[cid]; prev, c = chunks[k - 1], chunks[k]
    print(f"\n{prev['chunk_id']} ends   : …{prev['text'][-n:]!r}")
    print(f"{cid} starts : {c['text'][:n]!r}")

print("=== starts lower-case ===")
for cid in [c["chunk_id"] for c in checks["starts lower-case (cut mid-sentence?)"]][:5]:
    boundary(cid)

print("\n=== lone heading fragment ===")
for c in checks["has a lone heading fragment"]:
    frag = [p for p in c["text"].split("\n\n") if re.fullmatch(r"(Annexure|Chapter|[IVX]+|\d+)", p.strip())]
    print(f"{c['chunk_id']}: {frag} · …{c['text'][-120:]!r}")

print("\n=== ends without punctuation (5 examples) ===")
for c in checks["ends without . : ; ? ! )"][:5]:
    print(f"{c['chunk_id']}: …{c['text'][-90:]!r}")

=== starts lower-case ===

S099-03 ends   : …'ools.\n\no Lead data driven quality improvement activities.\n\no Effective coaching and engaging of all hospital staff towards delivering effective health services.'
S099-04 starts : 'o Effective coaching and engaging of all hospital staff towards delivering effective health services.\n\no Optimal utilization of available resources including HR'

S099-04 ends   : …'uld be made.\n\no Hospital staff participated in learning sessions for clinical training, data analysis, progress review, cross learning, experience sharing, etc.'
S099-05 starts : 'o Hospital staff participated in learning sessions for clinical training, data analysis, progress review, cross learning, experience sharing, etc.\n\n- Staff Moti'

L10-08 ends   : …'have already done.\n\n- in order to operationalize FRUs, 4 months quality EmOC and LSAS training skills training for MBBS doctors also needs to be explored fully.'
L10-09 starts : '- in order to operationalize FRUs, 4 mon